In [1]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over. This repository uses the flat layout
#: `<repo>/notebooks` -> `<repo>`, so the depth is 0.
LOCAL_ROOT_DEPTH = 0


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


# Synthetic selection pilot — the three candidate identifiability constraints of r05

Executed report of the pilot r05 § "Falsifiable claim and assumptions" declares
under "Constraint selection (synthetic study)". **The results are PROVISIONAL:
they feed the deliberation that closes the candidate selection in r06, and no
candidate is the design until it does** — r05: "Until the study reports, no
candidate is the design."

What runs, identically for every candidate (same protocol, same data per
scenario and seed):

- the declared synthetic world (K = 4, R = 4, 2 regions, N = 4000, the
  configuration in `__implementation__["premises"]`), three scenarios —
  region-varying reliability, per-category noise, correlated annotator bias;
- the warm-up estimation shared by everybody: Dawid-Skene EM → ρ̂_k → C̃_k by
  Eq. (4), plus C1's budget ρ_{r,k} = 1 − T̂_r[k,k] from the same EM;
- φ — the declared per-(annotator, category, region) logit table — trained
  against exactly one of r05's three objectives with the segmentation held
  fixed, never by (Ec. 1): **C1** (Ec. 5), **C2** (Ec. 6) over the declared λ
  grid {0.05, 0.2, 0.8, 3.2}, **C3** (Ec. 7);
- r05's four selection metrics plus the case-A sensor F_k(1/K).

The pilot runs 5 seeds per cell, below the declared verdict floor of 30: the
tables print, no verdict is granted, and the conclusions cell reports which
metrics separate candidates clearly and which do not.

Re-run headlessly from the repository root:

```
PATH="$PWD/.venv/bin:$PATH" .venv/bin/python -m jupyter nbconvert \
  --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=3600 notebooks/selection.ipynb
```


In [2]:
import json
import sys
import time

import torch

sys.path.insert(0, str(ROOT / "src"))

# The fingerprint of the code this report ran against — `report_digest` is the
# single implementation; `verify` recomputes and compares it.
from CrowdSegmenter_CT_GCE import report_digest
print(report_digest.stamp())

from CrowdSegmenter_CT_GCE.selection_study import (
    LAMBDA_GRID, PILOT_SEEDS, SCENARIOS, VERDICT_FLOOR_SEEDS, StudyConfig,
    floor_sensor, generate_world, run_cell, run_unconstrained_reference,
    scenario_accuracy, true_reliability_table, truth_gate_correlation)

CONFIG = StudyConfig()
CANDIDATE_ROWS = [("C1", None)] + [("C2", lam) for lam in LAMBDA_GRID] + [("C3", None)]
print("declared configuration:", json.dumps(CONFIG.as_dict()))
print("scenarios:", list(SCENARIOS))
print("seeds:", CONFIG.seeds, "| verdict floor:", VERDICT_FLOOR_SEEDS)


SOURCES-SHA256 b6cac53b858f34379fd46f06e77eb24c6f3eb37f373d42a877bc797252e994b3
declared configuration: {"n_classes": 4, "n_annotators": 4, "n_regions": 2, "n_pixels": 4000, "coverage": 0.8, "prior": [0.4, 0.3, 0.2, 0.1], "region_split": [0.7, 0.3], "model_error": [0.1, 0.3], "model_eps": 0.15, "lambda_grid": [0.05, 0.2, 0.8, 3.2], "seeds": [0, 1, 2, 3, 4], "branchSteps": 1500, "branchLr": 0.05, "qUnconstrained": 0.7, "collapse": {"saturationFraction": 0.9, "boundary": 0.02, "stdFloor": 0.01}, "pilotSeeds": 5, "verdictFloorSeeds": 30}
scenarios: ['region-varying', 'per-category', 'correlated-bias']
seeds: (0, 1, 2, 3, 4) | verdict floor: 30


## Timing one run before committing the grid

The estimate is cheaper than the cost and belongs here, not in a sentence
somewhere else. The figure below is measured, never guessed.

In [3]:
t0 = time.time()
probe = run_cell(CONFIG, SCENARIOS[0], CONFIG.seeds[0], "C2", lam=LAMBDA_GRID[2])
one_run_s = time.time() - t0
n_train = len(SCENARIOS) * len(CONFIG.seeds) * len(CANDIDATE_ROWS)
n_ref = len(SCENARIOS) * len(CONFIG.seeds)
print(f"one full run (world + warm-up EM + training + metrics): {one_run_s:.2f} s")
print(f"grid: {n_train} training runs + {n_ref} analytic references "
      f"~ {one_run_s * n_train / 60:.1f} min of training")


one full run (world + warm-up EM + training + metrics): 7.63 s
grid: 90 training runs + 15 analytic references ~ 11.4 min of training


## The grid

Outermost loop: the scenario. Then the seed — a repetition varies the material
it draws, not merely an initialization. Every (scenario, seed) world is
generated once and handed to every candidate unchanged.

In [4]:
records = []
sensor_by_scenario = {}
truth_corr_by_scenario = {}
truth_tables = {}

for scenario in SCENARIOS:
    world_probe = generate_world(CONFIG, scenario, CONFIG.seeds[0])
    truth_tables[scenario] = {
        "lam_gen": world_probe["lam_gen"].tolist(),
        "lam_true": world_probe["lam_true"].tolist(),
    }
    for seed in CONFIG.seeds:
        world = generate_world(CONFIG, scenario, seed)  # regenerated, deterministic
        for candidate, lam in CANDIDATE_ROWS:
            records.append(run_cell(CONFIG, scenario, seed, candidate, lam))
        records.append(run_unconstrained_reference(CONFIG, scenario, seed))
    # The warm-up sensor and the truth's gate correlation: properties of the
    # world and the warm-up, identical for every candidate.
    per_seed_ffloor = []
    per_seed_truth_corr = []
    for seed in CONFIG.seeds:
        world = generate_world(CONFIG, scenario, seed)
        from CrowdSegmenter_CT_GCE.selection_study import estimate_thresholds
        thresholds = estimate_thresholds(world, CONFIG)
        per_seed_ffloor.append(floor_sensor(world, thresholds))
        per_seed_truth_corr.append(truth_gate_correlation(CONFIG, scenario, seed))
    sensor_by_scenario[scenario] = per_seed_ffloor
    truth_corr_by_scenario[scenario] = per_seed_truth_corr

out_dir = ROOT / "results" / "selection"
out_dir.mkdir(parents=True, exist_ok=True)
record = {
    "what": "synthetic selection pilot -- provisional, feeds the r06 deliberation; "
            "no candidate is the design until the deliberation publishes it",
    "revision": "research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r09.md",
    "digest": report_digest.stamp(),
    "config": CONFIG.as_dict(),
    "candidateRows": [{"candidate": c, "lambda": l} for c, l in CANDIDATE_ROWS],
    "truthTables": truth_tables,
    "truthGateCorrelation": truth_corr_by_scenario,
    "floorSensor": sensor_by_scenario,
    "runs": records,
}
(out_dir / "selection_pilot.json").write_text(json.dumps(record, indent=1))
print(f"{len(records)} runs recorded -> {out_dir / 'selection_pilot.json'}")


105 runs recorded -> /home/lucas/papers/ct-gce/implementations/CrowdSegmenter-CT-GCE/results/selection/selection_pilot.json


## Comparative table — three candidates × four metrics × three scenarios

Means over the 5 seeds with the standard deviation beside them; the collapse
rate is the fraction of seeds that collapsed. `unconstrained` is not a
candidate: it is Eq. (2)'s analytic optimum, the pure truncation gate r05 warns
the branch would re-learn without a constraint, printed as the floor every row
is read against. Provisional numbers — no verdict is granted at 5 seeds.

In [5]:
from collections import defaultdict

def mean(xs):
    xs = [x for x in xs if x == x]
    return sum(xs) / len(xs) if xs else float("nan")

def std(xs):
    xs = [x for x in xs if x == x]
    if len(xs) < 2:
        return float("nan")
    m = sum(xs) / len(xs)
    return (sum((x - m) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5

def row_of(run):
    return (run["scenario"], run["candidate"], run["lambda"])

groups = defaultdict(list)
for run in records:
    groups[row_of(run)].append(run)

ORDER = list(CANDIDATE_ROWS) + [("unconstrained", None)]

def cell(mean_, std_):
    return f"{mean_:.3f} ± {std_:.3f}"

lines = ["| scenario | candidate | λ | recovery error ↓ | gate correlation | collapse rate | pruning precision ↑ | pruning recall ↑ |",
         "| --- | --- | --- | --- | --- | --- | --- | --- |"]
for scenario in SCENARIOS:
    for candidate, lam in ORDER:
        runs = groups[(scenario, candidate, lam)]
        assert runs, (scenario, candidate, lam)
        rec = [r["metrics"]["recoveryError"] for r in runs]
        gc = [r["metrics"]["gateCorrelation"] for r in runs]
        col = [1.0 if r["metrics"]["collapse"]["collapsed"] else 0.0 for r in runs]
        pr = [r["metrics"]["pruning"]["precision"] for r in runs]
        rc = [r["metrics"]["pruning"]["recall"] for r in runs]
        label = candidate if lam is None else f"C2 (λ={lam})"
        lines.append("| " + " | ".join([
            scenario, label, "—" if lam is None else f"{lam}",
            cell(mean(rec), std(rec)),
            cell(mean(gc), std(gc)),
            f"{mean(col):.2f}",
            cell(mean(pr), std(pr)),
            cell(mean(rc), std(rc)),
        ]) + " |")
    lines.append("| | | | | | | | |")
table = "\n".join([l for l in lines if not l.endswith("| | | | | | | | |")])
print(table)


| scenario | candidate | λ | recovery error ↓ | gate correlation | collapse rate | pruning precision ↑ | pruning recall ↑ |
| --- | --- | --- | --- | --- | --- | --- | --- |
| region-varying | C1 | — | 0.135 ± 0.004 | 0.119 ± 0.035 | 0.00 | 0.372 ± 0.017 | 0.351 ± 0.017 |
| region-varying | C2 (λ=0.05) | 0.05 | 0.032 ± 0.004 | 0.380 ± 0.014 | 0.00 | 0.495 ± 0.005 | 0.467 ± 0.007 |
| region-varying | C2 (λ=0.2) | 0.2 | 0.048 ± 0.002 | 0.381 ± 0.015 | 0.00 | 0.495 ± 0.005 | 0.467 ± 0.007 |
| region-varying | C2 (λ=0.8) | 0.8 | 0.241 ± 0.006 | 0.379 ± 0.014 | 0.00 | 0.495 ± 0.005 | 0.467 ± 0.007 |
| region-varying | C2 (λ=3.2) | 3.2 | 0.630 ± 0.001 | 0.381 ± 0.014 | 0.00 | 0.495 ± 0.006 | 0.467 ± 0.007 |
| region-varying | C3 | — | 0.032 ± 0.004 | 0.379 ± 0.014 | 0.00 | 0.495 ± 0.005 | 0.467 ± 0.007 |
| region-varying | unconstrained | — | 0.322 ± 0.005 | 1.000 ± 0.000 | 1.00 | 0.647 ± 0.012 | 0.610 ± 0.013 |
| per-category | C1 | — | 0.077 ± 0.008 | 0.200 ± 0.015 | 0.00 | 0.298 ± 0.014 |

### The truth's own correlation with the gate

The floor metric 2 can fall to before a candidate is suspected of merely
reproducing the gate: even a perfect recovery still correlates with the gate,
because accurate annotators agree with an accurate model more often than not.

In [6]:
lines = ["| scenario | truth↔gate correlation | unconstrained (the gate itself) |",
         "| --- | --- | --- |"]
for scenario in SCENARIOS:
    lines.append(f"| {scenario} | {mean(truth_corr_by_scenario[scenario]):.3f} | 1.000 |")
print("\n".join(lines))


| scenario | truth↔gate correlation | unconstrained (the gate itself) |
| --- | --- | --- |
| region-varying | 0.173 | 1.000 |
| per-category | 0.233 | 1.000 |
| correlated-bias | 0.209 | 1.000 |


## The case-A sensor — F_k(1/K) per category after the warm-up

How much the 1/K clip of Eq. (4) discards beyond the noise estimate, and
where: `F_k(1/K)` is the fraction of category-k annotated confidences at or
below the floor; where it exceeds ρ̂_k the clip is discarding more than the
estimate asks and the difference is the over-discarding, by category. A
property of the world and the warm-up — identical for every candidate.

In [7]:
lines = ["| scenario | category | ρ̂_k | C̃_k | F_k(1/K) | over-discarding |",
         "| --- | --- | --- | --- | --- | --- |"]
for scenario in SCENARIOS:
    per_seed = sensor_by_scenario[scenario]
    k = len(per_seed[0]["fFloor"])
    for cat in range(k):
        rho = mean([s["rhoHat"][cat] for s in per_seed])
        cut = mean([s["cutoffs"][cat] for s in per_seed])
        ff = mean([s["fFloor"][cat] for s in per_seed])
        over = mean([s["overDiscard"][cat] for s in per_seed])
        lines.append(f"| {scenario} | {cat} | {rho:.3f} | {cut:.3f} | {ff:.3f} | {over:+.3f} |")
print("\n".join(lines))


| scenario | category | ρ̂_k | C̃_k | F_k(1/K) | over-discarding |
| --- | --- | --- | --- | --- | --- |
| region-varying | 0 | 0.115 | 0.250 | 0.225 | +0.110 |
| region-varying | 1 | 0.166 | 0.250 | 0.258 | +0.092 |
| region-varying | 2 | 0.251 | 0.250 | 0.341 | +0.090 |
| region-varying | 3 | 0.418 | 0.250 | 0.484 | +0.066 |
| per-category | 0 | 0.143 | 0.250 | 0.262 | +0.120 |
| per-category | 1 | 0.173 | 0.250 | 0.277 | +0.104 |
| per-category | 2 | 0.260 | 0.250 | 0.370 | +0.109 |
| per-category | 3 | 0.615 | 0.589 | 0.594 | -0.021 |
| correlated-bias | 0 | 0.149 | 0.250 | 0.255 | +0.106 |
| correlated-bias | 1 | 0.195 | 0.250 | 0.294 | +0.099 |
| correlated-bias | 2 | 0.252 | 0.250 | 0.393 | +0.141 |
| correlated-bias | 3 | 0.390 | 0.250 | 0.541 | +0.151 |


## What the pilot separates, and what it leaves open

Generated from the record, never typed: two rows are read as separating on a
metric when their means differ by more than the combined standard error over
the seeds. **No verdict is granted**: the pilot runs 5 seeds and the declared
verdict floor is 30 — the numbers are provisional and feed the r06
deliberation, which closes the selection.

In [8]:
import math

def se(xs):
    s = std(xs)
    n = len([x for x in xs if x == x])
    return s / math.sqrt(n) if n and s == s else float("nan")

METRICS = [("recoveryError", "recovery error", "lower"),
           ("gateCorrelation", "gate correlation", "lower"),
           ("pruningP", "pruning precision", "higher"),
           ("pruningR", "pruning recall", "higher")]

def metric_values(scenario, candidate, lam, key):
    runs = groups[(scenario, candidate, lam)]
    if key == "pruningP":
        return [r["metrics"]["pruning"]["precision"] for r in runs]
    if key == "pruningR":
        return [r["metrics"]["pruning"]["recall"] for r in runs]
    return [r["metrics"][key] for r in runs]

print("Separation diagnostics (candidate rows only; the unconstrained "
      "reference is not a candidate):")
for scenario in SCENARIOS:
    for key, label, direction in METRICS:
        stats = {}
        for candidate, lam in CANDIDATE_ROWS:
            vals = metric_values(scenario, candidate, lam, key)
            stats[(candidate, lam)] = (mean(vals), se(vals))
        pairs = []
        names = list(stats)
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                m1, s1 = stats[names[i]]
                m2, s2 = stats[names[j]]
                if s1 != s1 or s2 != s2:
                    continue
                gap = abs(m1 - m2)
                combined = math.sqrt(s1 ** 2 + s2 ** 2)
                if combined > 0 and gap > combined:
                    better = names[i] if (
                        (m1 < m2) if direction == "lower" else (m1 > m2)) else names[j]
                    def nm(pair):
                        return pair[0] if pair[1] is None else f"{pair[0]}(λ={pair[1]})"
                    pairs.append(f"{nm(names[i])} vs {nm(names[j])} "
                                 f"(gap {gap:.3f} > SE {combined:.3f}; "
                                 f"{nm(better)} in the {direction} direction)")
        if pairs:
            print(f"- {scenario} / {label}: " + "; ".join(pairs))
        else:
            print(f"- {scenario} / {label}: indistinguishable at 5 seeds "
                  f"— no pair of candidate rows separates beyond its combined "
                  f"standard error")
print()
print(f"NO VERDICT: {len(CONFIG.seeds)} seeds per cell, below the declared "
      f"verdict floor of {VERDICT_FLOOR_SEEDS}. The tables above are "
      f"provisional evidence for the r06 deliberation, not a selection.")


Separation diagnostics (candidate rows only; the unconstrained reference is not a candidate):
- region-varying / recovery error: C1 vs C2(λ=0.05) (gap 0.103 > SE 0.003; C2(λ=0.05) in the lower direction); C1 vs C2(λ=0.2) (gap 0.087 > SE 0.002; C2(λ=0.2) in the lower direction); C1 vs C2(λ=0.8) (gap 0.106 > SE 0.003; C1 in the lower direction); C1 vs C2(λ=3.2) (gap 0.495 > SE 0.002; C1 in the lower direction); C1 vs C3 (gap 0.103 > SE 0.003; C3 in the lower direction); C2(λ=0.05) vs C2(λ=0.2) (gap 0.016 > SE 0.002; C2(λ=0.05) in the lower direction); C2(λ=0.05) vs C2(λ=0.8) (gap 0.209 > SE 0.003; C2(λ=0.05) in the lower direction); C2(λ=0.05) vs C2(λ=3.2) (gap 0.598 > SE 0.002; C2(λ=0.05) in the lower direction); C2(λ=0.2) vs C2(λ=0.8) (gap 0.193 > SE 0.003; C2(λ=0.2) in the lower direction); C2(λ=0.2) vs C2(λ=3.2) (gap 0.582 > SE 0.001; C2(λ=0.2) in the lower direction); C2(λ=0.2) vs C3 (gap 0.016 > SE 0.002; C3 in the lower direction); C2(λ=0.8) vs C2(λ=3.2) (gap 0.389 > SE 0.003; C2(